## Chapter 5 – Implementing Quantum-Safe Security

This notebook explores how modern cryptography is adapting to the arrival of quantum computing. The focus is on the mathematical foundations of post-quantum cryptography and the practical engineering trade-offs involved in deploying quantum-safe security.

The exercises begin with a series of lattice-based examples that build intuition for high-dimensional geometry, noise, and the Closest Vector Problem (CVP). Using the recurring characters Daisy and Donald, the notebook demonstrates how information that is easy to recover with private knowledge can become difficult to recover from public information alone.

Next, the notebook introduces `Learning With Errors (LWE)`, showing how carefully structured noise transforms an ordinary algebra problem into a much harder one. Additional experiments explore how search complexity grows with dimension and how distance relationships change in high-dimensional spaces.

The final section moves from toy examples to production cryptography. Using the `pqcrypto` library, the notebook performs a complete `ML-KEM` key establishment exchange, compares post-quantum and classical key sizes, and visualizes the network and infrastructure trade-offs associated with deploying quantum-safe cryptography at internet scale.

### Code 5-0: Generate Lattice Figures 5-1 and 5-2

This code generates the two lattice visualizations used in this chapter. Figure 5-1 shows how a valid lattice point can be shifted by injected noise, producing a nearby coordinate that is no longer exactly on the lattice. Figure 5-2 illustrates how the same lattice can be described using different bases, including a clean basis and a skewed basis. Together, these figures introduce the ideas of noise, lattice geometry, and alternative coordinate descriptions that form the foundation of modern lattice-based cryptography.

#### Figure 5-1 diagram generation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# === Figure 5-1: Noise Shifts the Message Off the Grid ===

valid, noisy = np.array([3.0, 2.0]), np.array([4.15, 3.15])
fig, ax = plt.subplots(figsize=(11, 7))

# Lattice grid and points
for x in range(1, 7): ax.axvline(x, color="gray", ls=(0, (4, 4)), lw=1.2, alpha=.65)
for y in range(1, 6): ax.axhline(y, color="gray", ls=(0, (4, 4)), lw=1.2, alpha=.65)

xs, ys = np.meshgrid(range(1, 7), range(1, 6))
ax.scatter(xs, ys, s=150, color="#D9D9D9", edgecolor="#555555", zorder=3)
ax.scatter(*valid, s=390, color="royalblue", edgecolor="black", lw=1.8, zorder=7)
ax.scatter(*noisy, s=470, color="crimson", edgecolor="black", lw=1.8, zorder=8)

# Squiggly noise path
d = noisy - valid
u, t = d / np.linalg.norm(d), np.linspace(0, 1, 160)
n = np.array([-u[1], u[0]])
path = valid + t[:, None] * d + .045 * np.sin(8 * np.pi * t)[:, None] * n
path[-12:] = valid + t[-12:, None] * d

ax.plot(path[:, 0], path[:, 1], color="black", lw=2, zorder=5)
ax.annotate("", xy=noisy, xytext=path[-15],
            arrowprops=dict(arrowstyle="-|>", color="black", lw=2,
                            mutation_scale=18), zorder=6)

# Labels
box = dict(boxstyle="round,pad=.35", fc="white", ec="black", lw=1, alpha=.95)
red_box = {**box, "ec": "crimson"}

labels = [
    ("Valid lattice point", valid, (.85, 2.0), "black", box, 16),
    ("Noisy transmitted\ncoordinate", noisy, (5.05, 4.35), "black", box, 18),
    ("Injected noise", (3.55, 2.72), (2.3, 3.95), "crimson", red_box, 6)
]

for text, xy, xytext, color, bbox, shrinkB in labels:
    ax.annotate(text, xy=xy, xytext=xytext, fontsize=15, color=color,
                ha="left", va="center", bbox=bbox,
                arrowprops=dict(arrowstyle="-|>", color=color, lw=1.6,
                                mutation_scale=18, shrinkA=4,
                                shrinkB=shrinkB),
                zorder=10)

# Axes and layout
for spine in ax.spines.values(): spine.set_visible(False)

ax.set(xlim=(0, 6.5), ylim=(0, 5.6), aspect="equal",
       xticks=range(0, 7), yticks=range(0, 6))
ax.tick_params(length=0, labelsize=16, pad=8)

axis_arrow = dict(arrowstyle="-|>", color="black", lw=2.2,
                  mutation_scale=24, shrinkA=0, shrinkB=0)

ax.annotate("", xy=(6.45, 0), xytext=(0, 0), arrowprops=axis_arrow)
ax.annotate("", xy=(0, 5.55), xytext=(0, 0), arrowprops=axis_arrow)
ax.text(6.48, -.18, r"$x$", fontsize=24, ha="center", va="top")
ax.text(-.16, 5.58, r"$y$", fontsize=24, ha="right", va="center")

plt.tight_layout()
plt.savefig("figure_5_1_noise_lattice_xy.png", dpi=300, bbox_inches="tight")
plt.show()

#### Figure 5-2 diagram generation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch

# === Code 5-2: Same Lattice, Different Bases ===

pts, target = np.array([(x, y) for x in range(-3, 4) for y in range(-3, 4)]), np.array([3, 2])
box = dict(fc="white", ec="none", alpha=.82, pad=.1)

views = [
    ("Clean basis", "Short, perpendicular vectors", "Private-key style view",
     np.array([[1, 0], [0, 1]]), [(0, .28), (3.10, .28), (3.10, 2)],
     (np.array([.52, -.48]), np.array([-.62, .42]))),

    ("Skewed basis", "Longer, skewed vectors", "Public-key style view",
     np.array([[2, 1], [1, 1]]), [(0.08, .12), (2.08, 1.12), (3.08, 2.12)],
     (np.array([.55, .08]), np.array([-.42, .30])))
]

def add_arrow(ax, a, b, color, lw=2.7, dashed=False, alpha=1, z=8):
    ax.add_patch(FancyArrowPatch(
        a, b, arrowstyle="-|>", mutation_scale=17, linewidth=lw,
        linestyle=(0, (5, 4)) if dashed else "solid",
        color=color, alpha=alpha, zorder=z))

fig, axes = plt.subplots(1, 2, figsize=(12, 5.6))

for ax, (title, subtitle, footer, basis, path, offsets) in zip(axes, views):
    b1, b2 = basis

    ax.scatter(pts[:, 0], pts[:, 1], s=85, color="#D9D9D9", edgecolor="#555555", zorder=3)
    ax.scatter(*target, s=320, color="#F2C300", edgecolor="black", linewidth=1.8, zorder=10)
    ax.axhline(0, color="black", lw=1.8, zorder=2)
    ax.axvline(0, color="black", lw=1.8, zorder=2)
    ax.grid(True, linestyle=(0, (4, 4)), color="gray", lw=1, alpha=.22)

    for a, b in zip(path[:-1], path[1:]):
        add_arrow(ax, a, b, "dimgray", lw=2.3, dashed=True, alpha=.82, z=7)

    for vec, color, label, offset in [
        (b1, "royalblue", r"$b_1$", offsets[0]),
        (b2, "crimson", r"$b_2$", offsets[1])
    ]:
        add_arrow(ax, (0, 0), vec, color, lw=3.6, z=9)
        ax.text(*(vec + offset), label, fontsize=22, color=color,
                fontweight="bold", ha="center", va="center", bbox=box, zorder=11)

    ax.text(.5, 1.10, title, transform=ax.transAxes, ha="center", fontsize=20, color="#1155CC")
    ax.text(.5, 1.04, subtitle, transform=ax.transAxes, ha="center", fontsize=12)
    ax.text(.5, -.12, footer, transform=ax.transAxes, ha="center",
            fontsize=12, color="dimgray", style="italic")

    ax.set(xlim=(-3.75, 3.75), ylim=(-3.75, 3.75),
           aspect="equal", xticks=range(-3, 4), yticks=range(-3, 4))
    ax.tick_params(length=0, labelsize=12)
    for spine in ax.spines.values(): spine.set_visible(False)

plt.subplots_adjust(wspace=.42, top=.86, bottom=.18)
plt.savefig("figure_5_2_lattice_bases.png", dpi=300, bbox_inches="tight")
plt.show()

### Code 5-1: Public and Private Views of a Toy Lattice

This example introduces the core intuition behind lattice-based cryptography.
Daisy places a message on a lattice, adds a small amount of noise, and then
recovers the original point using information that only she possesses.
Donald sees only the public lattice and must solve a Closest Vector Problem
(CVP) by searching nearby lattice points. Compare the work performed by each
person and consider how that effort changes as lattices grow in dimension.

In [ ]:
import numpy as np
import pandas as pd

# === Code 5-1: Public and Private Views of a Toy Lattice ===

# Step 1: Define the private (clean) and public (skewed) bases.
private_basis = np.array([[1, 0], [0, 1]])   # Daisy's map
public_basis  = np.array([[2, 1], [1, 1]])   # Public map

# Step 2: Place a message on the lattice and add calibrated noise.
message_point     = np.array([3, 2])                 # Daisy's secret location
rng               = np.random.default_rng(seed=42)
noise             = rng.uniform(-0.45, 0.45, size=2) # Small random offset
transmitted_point = message_point + noise            # What Donald sees

# Step 3: Daisy recovers by subtracting the noise she added.
daisy_recovered = np.round(transmitted_point - noise).astype(int)

# Step 4: Donald brute-forces the Closest Vector Problem (CVP).
def nearest_lattice_point(target, basis, span=6, verbose=False):

    # Define a search window around the target.
    coords = np.arange(-span, span + 1)

    # Generate candidate recipes within that window.
    recipes = np.array(
        np.meshgrid(coords, coords)
    ).T.reshape(-1, 2)

    # Convert recipes into lattice points.
    points = recipes @ basis

    # Measure the distance to each candidate.
    distances = np.linalg.norm(points - target, axis=1)

    # Select the closest match.
    best = np.argmin(distances)

    if verbose:
        nearby = distances < 2.0
        for p, d in zip(points[nearby], distances[nearby]):
            print(f"  checked {p.tolist()} -> distance {d:.3f}")

    return points[best], recipes[best], distances[best], len(distances)

donald_point, _, donald_dist, iters = nearest_lattice_point(
    transmitted_point,
    public_basis,
    verbose=True
)

# Step 5: Compare outcomes.
print(f"\nMessage point:     {message_point.tolist()}")
print(f"Transmitted point: {transmitted_point.round(3).tolist()}")

actor_rows = [
    {
        "Actor": "Daisy",
        "Information Available": "Noise vector",
        "Method": "Subtract known noise",
        "Result": daisy_recovered.tolist(),
        "Work Required": "1 operation"
    },
    {
        "Actor": "Donald",
        "Information Available": "Public lattice only",
        "Method": "Brute-force CVP search",
        "Result": donald_point.tolist(),
        "Work Required": f"{iters} candidate checks"
    },
]

pd.DataFrame(actor_rows)

### Code 5-2: CVP Search Cost Across Dimensions

This example compares Daisy's recovery process with Donald's
brute-force Closest Vector Problem (CVP) search as the number of
dimensions increases. Daisy's workload remains nearly constant because
she knows the noise that was added. Donald must examine an ever-growing
number of candidate lattice points. Watch how quickly the search space
expands and consider what happens when the lattice grows from a simple
toy example to the high-dimensional spaces used in modern cryptography.

In [ ]:
import itertools
import time
import numpy as np
import pandas as pd

# === Code 5-2: CVP Search Cost Across Dimensions ===

TIMEOUT_SECONDS = 10


def make_lattice(dims, seed=42):
    """Generate a random lattice basis."""
    rng = np.random.default_rng(seed)
    return rng.integers(1, 5, size=(dims, dims))


def recover_with_noise(transmitted, noise):
    """Daisy removes the known noise."""
    return np.round(transmitted - noise).astype(int)


def brute_force_cvp(target, basis, span=3):
    """Donald checks every candidate point in a search window."""
    values = range(-span, span + 1)
    best_point, best_dist = None, float("inf")
    start = time.time()

    for recipe in itertools.product(values, repeat=basis.shape[0]):
        if time.time() - start > TIMEOUT_SECONDS:
            return best_point, time.time() - start, False

        point = np.array(recipe) @ basis
        dist = np.linalg.norm(point - target)

        if dist < best_dist:
            best_point, best_dist = point, dist

    return best_point, time.time() - start, True


dimensions, span = [2, 4, 6, 8, 10], 3
rows = []

for dims in dimensions:

    # Create a lattice, message, and noisy transmission.
    basis = make_lattice(dims)
    message = np.full(dims, 3)

    rng = np.random.default_rng(42)
    noise = rng.uniform(-0.45, 0.45, size=dims)
    transmitted = message + noise

    # Daisy's recovery path.
    t0 = time.time()
    recovered = recover_with_noise(transmitted, noise)
    recovery_time = time.time() - t0

    # Donald's brute-force search.
    _, search_time, completed = brute_force_cvp(
        transmitted, basis, span
    )

    candidates = (2 * span + 1) ** dims

    rows.append({
        "Dimensions": dims,
        "Candidates": f"{candidates:,}",
        "Recovery (sec)": f"{recovery_time:.6f}",
        "Search (sec)": f"{search_time:.3f}",
        "Recovery correct": np.array_equal(recovered, message),
        "Search finished": completed,
    })

    print(
        f"dim={dims:2d} | candidates={candidates:>15,} "
        f"| recovery={recovery_time:.6f}s "
        f"| search={search_time:.3f}s "
        f"| finished={completed}"
    )

pd.DataFrame(rows)

### Code 5-3: Learning With Errors in a Toy System

This example demonstrates the central idea behind Learning With Errors
(LWE). A secret value is combined with a small amount of carefully
chosen noise before being made public. An authorized user removes the
noise and recovers the correct secret, while an attacker attempts to
solve the same system without knowing the noise. Compare the results to
see how a small amount of noise can transform an ordinary algebra
problem into a much harder one.

In [ ]:
import numpy as np
import pandas as pd

# === Code 5-3: Learning With Errors in a Toy System ===

q = 17  # Small modulus; values wrap around like a clock.

# Public matrix A. Everyone can see this.
A = np.array([[2, 3], [1, 4]])

# Private values. Daisy knows these; Donald does not.
secret = np.array([3, 2])
noise = np.array([1, -1])

# LWE core: public result = matrix @ secret + small noise, mod q.
b = (A @ secret + noise) % q


def solve_mod_2x2(matrix, vector, modulus):
    """Solve a 2x2 linear system using modular arithmetic."""
    a, b, c, d = matrix.flatten()
    det = (a * d - b * c) % modulus
    inv_det = pow(int(det), -1, modulus)
    inverse = inv_det * np.array([[d, -b], [-c, a]])
    return (inverse @ vector) % modulus


# Authorized path: remove the known noise, then solve cleanly.
clean_b = (b - noise) % q
authorized_secret = solve_mod_2x2(A, clean_b, q)
authorized_check = (A @ authorized_secret) % q

# Attacker path: treat the noisy public result as if it were exact.
attacker_guess = np.linalg.solve(A, b)
attacker_secret = np.round(attacker_guess).astype(int)
attacker_check = (A @ attacker_secret) % q

rows = [
    {
        "View": "Authorized",
        "Knows noise": "Yes",
        "Method": "Remove noise, solve mod q",
        "Recovered secret": authorized_secret.tolist(),
        "Check": authorized_check.tolist(),
        "Correct": np.array_equal(authorized_secret, secret),
    },
    {
        "View": "Attacker",
        "Knows noise": "No",
        "Method": "Solve as if no noise exists",
        "Recovered secret": attacker_secret.tolist(),
        "Check": attacker_check.tolist(),
        "Correct": np.array_equal(attacker_secret, secret),
    },
]

print("Public matrix A:")
print(A)
print("\nPublic vector b:", b.tolist())
print("Private secret:", secret.tolist())
print("Private noise:", noise.tolist(), "\n")

pd.DataFrame(rows)

### Code 5-4: Distance Concentration Across Dimensions

This example generates Figure 5-3 by measuring the distance from random
points to their nearest lattice neighbors as the number of dimensions
increases. In low dimensions, nearby and faraway points are easier to
distinguish. As dimensions grow, those distances become increasingly
similar and the distributions tighten. This effect helps build intuition
for why high-dimensional lattice geometry becomes more difficult to
navigate and why distance alone becomes less useful as a guide.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# === Code 5-4: Distance Concentration Across Dimensions ===

dims_list = [2, 4, 8, 16, 32, 64]

# Colorful for ebook, ordered by luminance for grayscale print.
colors = ["#0B3C5D",  # dark blue
          "#328CC1",  # medium blue
          "#2E7D32",  # green
          "#F9A825",  # amber
          "#E65100",  # orange
          "#8E1B1B"]  # dark red

n_lattice, n_queries, bins = 500, 2000, 50
rows = []

# HD aspect ratio, but less wide than 15 x 6.
fig, ax = plt.subplots(figsize=(12, 6.75))

for dims, color in zip(dims_list, colors):
    rng = np.random.default_rng(dims)
    lattice = rng.integers(-5, 6, size=(n_lattice, dims))
    queries = rng.uniform(-3, 3, size=(n_queries, dims))

    nearest = np.array([
        np.min(np.linalg.norm(lattice - q, axis=1))
        for q in queries
    ])

    mean, std = nearest.mean(), nearest.std()
    rows.append({
        "Dimensions": dims,
        "Mean distance": round(mean, 3),
        "Std deviation": round(std, 3),
        "Relative spread": round(std / mean, 3)
    })

    ax.hist(nearest, bins=bins, density=True, alpha=0.60,
            color=color, edgecolor="none", label=f"{dims}D")
    ax.axvline(mean, color=color, linestyle="--", linewidth=2)

ax.set_xlabel("Distance to nearest lattice point", fontsize=16)
ax.set_ylabel("Density", fontsize=16)
ax.tick_params(axis="both", labelsize=14)
ax.legend(
    title="Dimensions",
    fontsize=16,
    title_fontsize=14,
    frameon=True,
    loc="upper right"
)

plt.tight_layout()
plt.savefig("figure_5_3_distance_concentration.png",
            dpi=300, bbox_inches="tight")
plt.show()

pd.DataFrame(rows)

### Code 5-5: ML-KEM in Practice

This example performs a complete ML-KEM key establishment exchange using
the same lattice-based algorithm now being deployed across the internet.
A public/private keypair is generated, a shared secret is encapsulated,
and the receiving party recovers the same secret through decapsulation.
The example also compares ML-KEM key and ciphertext sizes with classical
algorithms and generates Figure 5-5 to visualize the engineering
trade-offs involved in post-quantum deployment.


**Note:** Run the package installation cell that follows before executing
this example. The pqcrypto library provides the NIST post-quantum
algorithms used throughout this section.

In [ ]:
# === Install dependency ===
!pip install pqcrypto --quiet

In [ ]:
from pqcrypto.kem import ml_kem_512, ml_kem_768, ml_kem_1024
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

# ==========================================================
# Code 5-5: ML-KEM in Practice
# ==========================================================

# --- Font and color constants (adjust to taste) -----------
FONT_TITLE   = 20
FONT_AXIS    = 18
FONT_TICK    = 20
FONT_LEGEND  = 18
FONT_ANNOT   = 18

COLOR_CLASSIC  = "#c62828"    # red   for classical algorithms
COLOR_PQ       = "#1565c0"    # blue  for post-quantum algorithms
COLOR_LINE     = "#2e7d32"    # green for throughput line
COLOR_TLS      = "#666666"    # gray  for TLS reference line
COLOR_BBOX_FG  = "#2e7d32"    # annotation text color
COLOR_BBOX_BG  = "white"      # annotation box background
BBOX_ALPHA     = 0.85         # annotation box transparency
# ----------------------------------------------------------

# Step 1: Run a complete ML-KEM-768 key exchange.
pub, sec     = ml_kem_768.generate_keypair()  # generate public/private keypair
ct,  key_enc = ml_kem_768.encrypt(pub)        # encapsulate shared secret
key_dec      = ml_kem_768.decrypt(sec, ct)    # decapsulate shared secret

print("=== ML-KEM-768 Key Exchange ===")
print(f"Public key size:    {len(pub):>6} bytes")
print(f"Ciphertext size:    {len(ct):>6} bytes")
print(f"Shared secret size: {len(key_enc):>6} bytes")
print(f"Keys match:         {key_enc == key_dec}")

# Step 2: Collect sizes for all three ML-KEM security levels.
def kem_sizes(mod):
    pk, sk = mod.generate_keypair()
    c,  k  = mod.encrypt(pk)
    return len(pk), len(c), len(k)

pk512,  ct512,  ks512  = kem_sizes(ml_kem_512)
pk1024, ct1024, ks1024 = kem_sizes(ml_kem_1024)

rows = [
    ("RSA-2048",    "Classical",    294,      256,      32,           "No"),
    ("ECC P-256",   "Classical",    64,       96,       32,           "No"),
    ("ML-KEM-512",  "Post-quantum", pk512,    ct512,    ks512,        "Yes"),
    ("ML-KEM-768",  "Post-quantum", len(pub), len(ct),  len(key_enc), "Yes"),
    ("ML-KEM-1024", "Post-quantum", pk1024,   ct1024,   ks1024,       "Yes"),
]
cols = ["Algorithm", "Type", "Public key (B)",
        "Ciphertext (B)", "Shared key (B)", "Quantum safe"]
df   = pd.DataFrame(rows, columns=cols)
print("\n=== Algorithm Comparison ===")
print(df.to_string(index=False))

# Step 3: Visualize byte overhead and connection throughput.
def plot_comparison(df, save_path="figure_5_5_mlkem_overhead.png"):
    algorithms = df["Algorithm"].tolist()
    pub_sizes  = df["Public key (B)"].tolist()
    ct_sizes   = df["Ciphertext (B)"].tolist()
    totals     = [p + c for p, c in zip(pub_sizes, ct_sizes)]
    baseline   = 10_000_000              # 10 MB/s handshake bandwidth budget
    throughput = [baseline / t for t in totals]
    bar_colors = [COLOR_CLASSIC, COLOR_CLASSIC,
                  COLOR_PQ, COLOR_PQ, COLOR_PQ]
    x     = np.arange(len(algorithms))
    width = 0.5

    fig, ax1 = plt.subplots(figsize=(16, 8))
    fig.patch.set_facecolor("white")
    ax1.set_facecolor("white")

    # --- stacked bars: public key + ciphertext bytes ---
    ax1.bar(x, pub_sizes, width,
            color=bar_colors, alpha=0.95)
    ax1.bar(x, ct_sizes, width, bottom=pub_sizes,
            color=bar_colors, alpha=0.40)

    # --- bar total labels at the top of each bar ---
    bar_bbox = dict(boxstyle="round,pad=0.25",
                    facecolor="white", alpha=0.85,
                    edgecolor="#cccccc", linewidth=1.0)
    for xi, total in zip(x, totals):
        ax1.text(xi, total + 40, f"{total:,} B",
                 ha="center", va="bottom",
                 fontsize=FONT_ANNOT, color="#333333",
                 fontweight="bold", bbox=bar_bbox)

    # --- TLS reference line with label on the left ---
    ax1.axhline(y=1500, color=COLOR_TLS,
                linestyle="--", linewidth=1.5)
    ax1.text(0.01, 1520, "Typical TLS record (1500 B)",
             fontsize=FONT_ANNOT, color=COLOR_TLS,
             va="bottom", transform=ax1.get_yaxis_transform())

    ax1.set_ylabel("Total bytes (key + ciphertext)",
                   fontsize=FONT_AXIS)
    ax1.set_ylim(0, 3500)
    ax1.set_xticks(x)
    ax1.set_xticklabels(algorithms, fontsize=FONT_TICK)
    ax1.tick_params(axis="y", labelsize=FONT_TICK)

    # --- throughput line on secondary axis ---
    ax2 = ax1.twinx()
    ax2.plot(x, throughput, "o--",
             color=COLOR_LINE, linewidth=2.5, markersize=14)

    # annotations shifted left of each point
    bbox_props = dict(boxstyle="round,pad=0.3",
                      facecolor=COLOR_BBOX_BG,
                      alpha=BBOX_ALPHA,
                      edgecolor=COLOR_LINE,
                      linewidth=1.2)
    for xi, val in zip(x, throughput):
        ax2.annotate(
            f"{int(val):,}",
            xy=(xi, val),
            xytext=(-18, 8),
            textcoords="offset points",
            ha="right",
            fontsize=FONT_ANNOT,
            color=COLOR_BBOX_FG,
            fontweight="bold",
            bbox=bbox_props
        )

    ax2.set_ylabel("Connections per second\n(10 MB/s budget)",
                   fontsize=FONT_AXIS, color=COLOR_LINE)
    ax2.tick_params(axis="y", labelcolor=COLOR_LINE,
                    labelsize=FONT_TICK)

    # --- legend ---
    legend_els = [
        Patch(facecolor=COLOR_CLASSIC, alpha=0.95,
              label="Classical"),
        Patch(facecolor=COLOR_PQ,      alpha=0.95,
              label="Post-quantum"),
        Line2D([0], [0], color=COLOR_LINE, marker="o",
               linewidth=2.5, markersize=9,
               label="Connections/sec (right axis)"),
    ]
    ax1.legend(handles=legend_els, fontsize=FONT_LEGEND,
               loc="upper left", framealpha=1.0)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"Saved: {save_path}")

plot_comparison(df)